# Debugging notebook: Dephasing DMRG branch (BB codes)

This notebook is meant to debug the **DephasingDMRG** branch used when the logical MPS has more than 12 sites.

Main questions:
1. What basis string does DephasingDMRG actually return?
2. What is the estimated **maximum** basis amplitude \(\max_s |\langle s|\psi
angle|\) of the logical MPS?
3. Is the identity logical (all-zero string) in the MAP set?

Assumptions:
- You run this notebook from the mdopt repository root (so `examples/` exists).
- Your local environment has `mdopt`, `qecstruct`, `qldpc`, etc. installed or importable.


In [1]:
# --- Setup ---
import os
import sys
import logging

# Helpful for iterative debugging
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# If you run from repo root, this should be enough.
# Otherwise, set REPO_ROOT to the correct directory.
REPO_ROOT = os.path.abspath(os.getcwd())

# Ensure we can import the examples module (and mdopt if running in-place).
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# If needed, also add examples/ explicitly
EXAMPLES_ROOT = os.path.join(REPO_ROOT, 'examples')
if EXAMPLES_ROOT not in sys.path:
    sys.path.insert(0, EXAMPLES_ROOT)

print('REPO_ROOT:', REPO_ROOT)
print('EXAMPLES_ROOT:', EXAMPLES_ROOT)


REPO_ROOT: /Users/aleksandrberezutskii/mdopt/examples/decoding
EXAMPLES_ROOT: /Users/aleksandrberezutskii/mdopt/examples/decoding/examples


In [2]:
# --- Imports (from your decoding module) ---
from examples.decoding.decoding import (
    create_bb_code,
    generate_pauli_error_string,
    decode_css,
    # low-level utilities we will reuse to reconstruct the logical MPS
    pauli_to_mps,
    css_code_constraint_sites,
    css_code_logicals_sites,
    apply_bitflip_bias,
    apply_depolarising_bias,
)

from mdopt.mps.utils import create_custom_product_state, create_simple_product_state, inner_product
from mdopt.optimiser.utils import apply_constraints
from mdopt.optimiser.dephasing_dmrg import DephasingDMRG
from mdopt.optimiser.utils import XOR_LEFT, XOR_BULK, XOR_RIGHT, COPY_LEFT, SWAP

import numpy as np


In [3]:
# --- Experiment parameters (edit freely) ---
order_x = 6
order_y = 6
poly_a = 'x**3 + y + y**2'
poly_b = 'y**3 + x + x**2'

error_rate = 0.04
error_model = 'Bitflip'   # 'Bitflip' or 'Depolarising'

bias_prob = 1e-4
chi_max = 150
num_runs = 10

# Keep these at zero for now (as in your scripts).
tolerance = 0.0
cut = 0.0
renormalise = True
contraction_strategy = 'Optimised'
silent = False

# For reproducibility
seed = 777
rng = np.random.default_rng(seed)

In [4]:
# --- Build BB code and inspect parameters ---
bb_code = create_bb_code(order_x, order_y, poly_a, poly_b)

n = len(bb_code)
kx = bb_code.num_x_logicals()
kz = bb_code.num_z_logicals()
print('n =', n)
print('num_x_logicals =', kx)
print('num_z_logicals =', kz)
print('num_logicals =', kx + kz)

n = 72
num_x_logicals = 12
num_z_logicals = 12
num_logicals = 24


In [5]:
# --- Pick an error to debug ---
# Option A: generate a fresh random error
error = generate_pauli_error_string(n, error_rate, error_model=error_model, rng=rng)

# Option B: hardcode a specific error string
# error = 'I'*n

print('error:', error)
print('weight:', sum(c != 'I' for c in error))


error: IIIIIIIIIIIIIIIIIIIXIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
weight: 1


## Reconstruct the logical MPS explicitly

We replicate the internal steps of `decode_css` until we obtain `logical_mps`.
This lets us compute: 
- the amplitude of the all-zero logical, 
- the overlap found by DMRG, and 
- (for small systems) the exact max amplitude by densifying.


In [6]:
def build_logical_mps_css(code, error_str: str):
    """Return (logical_mps, meta) where logical_mps is the post-marginal MPS over logical qubits.

    This function mirrors the logic of decode_css up to the `logical_mps = ... .reverse()` line.
    """

    # Quick exit
    if error_str == 'I' * len(error_str):
        raise ValueError('Use a non-trivial error for debugging, otherwise logical_mps is trivial.')

    error_contains_x = 'X' in error_str
    error_contains_z = 'Z' in error_str

    erased_qubits = [i for i, e in enumerate(error_str) if e == 'E']
    if erased_qubits:
        raise NotImplementedError('This notebook focuses on the non-erasure branch.')

    # Convert to the MPS encoder alphabet (00/01/10/11)
    error_mps_str = pauli_to_mps(error_str)

    num_sites = 2 * len(code) + code.num_x_logicals() + code.num_z_logicals()
    num_logicals = code.num_x_logicals() + code.num_z_logicals()

    if len(error_mps_str) != num_sites - num_logicals:
        raise ValueError(f'error length {len(error_mps_str)} != expected {num_sites - num_logicals}')

    logicals_state = '+' * num_logicals
    state_string = logicals_state + error_mps_str

    # Right-canonical initial product state
    error_mps = create_custom_product_state(
        string=state_string,
        tolerance=tolerance,
        form='Right-canonical',
    )

    # Tensors and sites
    constraints_tensors = [XOR_LEFT, XOR_BULK, SWAP, XOR_RIGHT]
    logicals_tensors = [COPY_LEFT, XOR_BULK, SWAP, XOR_RIGHT]

    constraint_sites = css_code_constraint_sites(code)
    logicals_sites = css_code_logicals_sites(code)
    sites_to_bias = list(range(num_logicals, num_sites))

    # Bias
    if error_model == 'Bitflip':
        error_mps = apply_bitflip_bias(
            mps=error_mps,
            sites_to_bias=sites_to_bias,
            prob_bias_list=bias_prob,
        )
    else:
        error_mps = apply_depolarising_bias(
            mps=error_mps,
            sites_to_bias=sites_to_bias,
            prob_bias_list=bias_prob,
            renormalise=renormalise,
        )

    # Apply logical constraints
    if error_contains_x:
        error_mps = apply_constraints(
            error_mps,
            logicals_sites[0],
            logicals_tensors,
            chi_max=chi_max,
            cut=cut,
            renormalise=renormalise,
            silent=silent,
            strategy=contraction_strategy,
        )

    if error_contains_z:
        error_mps = apply_constraints(
            error_mps,
            logicals_sites[1],
            logicals_tensors,
            chi_max=chi_max,
            cut=cut,
            renormalise=renormalise,
            silent=silent,
            strategy=contraction_strategy,
        )

    # Apply check constraints
    if error_contains_x:
        error_mps = apply_constraints(
            error_mps,
            constraint_sites[0],
            constraints_tensors,
            chi_max=chi_max,
            cut=cut,
            renormalise=renormalise,
            silent=silent,
            strategy=contraction_strategy,
        )

    if error_contains_z:
        error_mps = apply_constraints(
            error_mps,
            constraint_sites[1],
            constraints_tensors,
            chi_max=chi_max,
            cut=cut,
            renormalise=renormalise,
            silent=silent,
            strategy=contraction_strategy,
        )

    # Marginalise out physical error sites -> leave only logical sites
    sites_to_marginalise = list(range(num_logicals, len(error_mps_str) + num_logicals))
    logical_mps = error_mps.marginal(
        sites_to_marginalise=sites_to_marginalise,
        renormalise=renormalise,
    ).reverse()

    meta = {
        'num_sites_total': num_sites,
        'num_logicals': num_logicals,
        'num_logical_sites': len(logical_mps),
        'error_contains_x': error_contains_x,
        'error_contains_z': error_contains_z,
    }

    return logical_mps, meta


In [7]:
logical_mps, meta = build_logical_mps_css(bb_code, error)
print(meta)
print('logical_mps num_sites:', len(logical_mps))


100%|██████████| 36/36 [00:31<00:00,  1.15it/s]

{'num_sites_total': 168, 'num_logicals': 24, 'num_logical_sites': 24, 'error_contains_x': True, 'error_contains_z': False}
logical_mps num_sites: 24


## Compute amplitudes

We compute:
- \(|\langle 0\cdots 0|\psi
angle|\) (identity logical amplitude),
- DMRG-selected product string \(|s_*
angle\) and \(|\langle s_*|\psi
angle|\),
- optional exact max amplitude if \(\#	ext{logical sites}\) is small enough.


In [8]:
def product_state_bitstring(mps_product) -> str:
    """Extract the computational basis bitstring from a product-state MPS.

    Assumes each site tensor has bond dims 1 (or effectively rank-1), so the local state
    is proportional to tensor[0, :, 0]. We take argmax over physical dimension (0/1).
    """
    bits = []
    for A in mps_product.tensors:
        # A is typically shape (Dl, d, Dr). For a product state Dl=Dr=1.
        v = A.reshape(-1)
        # If shape is (d,), use directly; else take the middle axis.
        if A.ndim == 3:
            local = A[0, :, 0]
        elif A.ndim == 2:
            # sometimes stored as (d,1) etc.
            local = v
        else:
            local = v
        b = int(np.argmax(np.abs(local)))
        bits.append(str(b))
    return ''.join(bits)


def basis_amplitude(logical_mps, bitstring: str) -> float:
    """Return |<bitstring|logical_mps>| for computational basis bitstring."""
    target = create_custom_product_state(bitstring, tolerance=0.0, form='Right-canonical')
    return float(abs(inner_product(target, logical_mps)))


num_logical_sites = len(logical_mps)

# Identity logical amplitude (all zeros)
zero_str = '0' * num_logical_sites
amp_zero = basis_amplitude(logical_mps, zero_str)
print('amp_zero =', amp_zero)


amp_zero = 0.015624999999999953


In [9]:
# --- Run Dephasing DMRG on the logical MPS ---
mps_start = create_simple_product_state(num_logical_sites, which='+')
engine = DephasingDMRG(
    mps=mps_start,
    mps_target=logical_mps,
    chi_max=chi_max,
    cut=cut,
    mode='LA',
    silent=silent,
)

engine.run(num_iter=num_runs)
mps_star = engine.mps  # product MPS returned by DephasingDMRG

bitstring_star = product_state_bitstring(mps_star)
amp_star = float(abs(inner_product(mps_star, logical_mps)))

print('bitstring_star:', bitstring_star)
print('amp_star      :', amp_star)
print('amp_zero      :', amp_zero)


100%|██████████| 10/10 [00:03<00:00,  3.04it/s]

bitstring_star: 110101111110000000000000
amp_star      : 0.01562500000000043
amp_zero      : 0.015624999999999953


In [10]:
# --- Check whether identity is MAP (within numerical tolerance) ---
# If DephasingDMRG is working as intended, amp_star is an estimate of max amplitude.
# The identity logical is MAP if amp_zero is within eps of amp_star.

rel_tol = 1e-9
abs_tol = 1e-12
eps = max(rel_tol * amp_star, abs_tol)

is_map_identity_est = (amp_zero >= amp_star - eps)
print('eps:', eps)
print('MAP(identity) [estimate]:', is_map_identity_est)


eps: 1.562500000000043e-11
MAP(identity) [estimate]: True


In [11]:
# --- Optional: exact max amplitude by densifying (only feasible for small #logical sites) ---
# This is the ground truth for comparing the DMRG-selected amplitude.

max_dense_sites = 26
if num_logical_sites <= max_dense_sites:
    dense = np.abs(logical_mps.dense(flatten=True, renormalise=renormalise, norm=2))
    i_max = int(np.argmax(dense))
    amp_max = float(dense[i_max])
    amp_id = float(dense[0])

    print('Exact amp_max:', amp_max)
    print('Exact amp_id :', amp_id)

    # Report the MAP bitstring
    bitstring_max = format(i_max, '0{}b'.format(num_logical_sites))
    print('Exact MAP bitstring:', bitstring_max)

    # Compare to DMRG
    print('DMRG amp_star / amp_max =', amp_star / amp_max if amp_max > 0 else np.nan)
else:
    print(f'Skipping densify: num_logical_sites={num_logical_sites} > {max_dense_sites}')


Exact amp_max: 0.01562500000000043
Exact amp_id : 0.01562499999999995
Exact MAP bitstring: 010001111110000000000000
DMRG amp_star / amp_max = 1.0
